**Variation 1**


In [ ]:
import cv2
import numpy as np
import os
import time
from google.colab import files

# =====================================================
# UPLOAD VIDEO
# =====================================================

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Uploaded:", video_path)


# =====================================================
# PARAMETERS
# =====================================================

MIN_TABLET_AREA = 1000

CHIP_THRESHOLD = 250

CAP_THRESHOLD = 180

kernel = np.ones((5,5), np.uint8)


# =====================================================
# PREPROCESS
# =====================================================

def preprocess(frame):

    blur = cv2.GaussianBlur(
        frame,
        (5,5),
        0
    )

    hsv = cv2.cvtColor(
        blur,
        cv2.COLOR_BGR2HSV
    )

    gray = hsv[:,:,2]

    return gray


# =====================================================
# SEGMENT TABLET
# =====================================================

def segment_tablet(gray):

    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours)==0:
        return None,None

    contour=max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(contour)<MIN_TABLET_AREA:
        return None,None

    clean=np.zeros_like(mask)

    cv2.drawContours(
        clean,
        [contour],
        -1,
        255,
        -1
    )

    return clean,contour


# =====================================================
# CHIP DETECTION
# =====================================================

def detect_chip(mask, contour):

    hull = cv2.convexHull(contour)

    hull_mask = np.zeros_like(mask)

    cv2.drawContours(
        hull_mask,
        [hull],
        -1,
        255,
        -1
    )

    chip = cv2.subtract(
        hull_mask,
        mask
    )

    chip_area = np.sum(chip>0)

    return chip, chip_area


# =====================================================
# CAP DETECTION
# =====================================================

def detect_cap(gray, mask):

    lap = cv2.Laplacian(
        gray,
        cv2.CV_64F
    )

    lap = np.uint8(
        np.abs(lap)
    )

    cap = cv2.bitwise_and(
        lap,
        lap,
        mask=mask
    )

    cap_area = np.sum(
        cap>35
    )

    return cap, cap_area


# =====================================================
# PROCESS FRAME
# =====================================================

def process(frame):

    gray = preprocess(frame)

    mask, contour = segment_tablet(gray)

    if contour is None:
        return frame

    chip_mask, chip_area = detect_chip(
        mask,
        contour
    )

    cap_mask, cap_area = detect_cap(
        gray,
        mask
    )

    tablet_area = np.sum(mask>0)

    chip_percent = (
        chip_area/tablet_area
    )*100

    cap_percent = (
        cap_area/tablet_area
    )*100

    defect=False
    label="GOOD"

    if chip_area>CHIP_THRESHOLD:

        defect=True
        label=f"CHIPPING {chip_percent:.2f}%"

    elif cap_area>CAP_THRESHOLD:

        defect=True
        label=f"CAPPING {cap_percent:.2f}%"

    color=(0,255,0)

    if defect:
        color=(0,0,255)

    cv2.drawContours(
        frame,
        [contour],
        -1,
        color,
        3
    )

    cv2.putText(
        frame,
        label,
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        color,
        2
    )

    return frame


# =====================================================
# VIDEO PROCESSING
# =====================================================

cap = cv2.VideoCapture(video_path)

fps = int(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)

w = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

h = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

output_path = "processed_tablet_video.mp4"

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (w,h)
)

count=0
start=time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    result = process(frame)

    writer.write(result)

    count+=1

    if count%20==0:

        elapsed=time.time()-start

        print(
            f"Processed {count} frames | "
            f"FPS={count/elapsed:.2f}"
        )

cap.release()

writer.release()


print("\nDone!")
print("Saved:", output_path)


# =====================================================
# DOWNLOAD
# =====================================================

files.download(
    output_path
)

Saving input1_crop.mp4 to input1_crop.mp4
Uploaded: input1_crop.mp4
Processed 20 frames | FPS=35.58
Processed 40 frames | FPS=39.68
Processed 60 frames | FPS=40.39
Processed 80 frames | FPS=40.98
Processed 100 frames | FPS=41.65
Processed 120 frames | FPS=42.13
Processed 140 frames | FPS=42.71
Processed 160 frames | FPS=43.11
Processed 180 frames | FPS=43.45
Processed 200 frames | FPS=43.80
Processed 220 frames | FPS=43.93
Processed 240 frames | FPS=44.15
Processed 260 frames | FPS=44.27
Processed 280 frames | FPS=44.42
Processed 300 frames | FPS=44.50
Processed 320 frames | FPS=44.63
Processed 340 frames | FPS=44.75
Processed 360 frames | FPS=44.81
Processed 380 frames | FPS=44.85
Processed 400 frames | FPS=44.90
Processed 420 frames | FPS=44.85
Processed 440 frames | FPS=44.82
Processed 460 frames | FPS=43.96
Processed 480 frames | FPS=43.00
Processed 500 frames | FPS=42.36
Processed 520 frames | FPS=41.72
Processed 540 frames | FPS=41.62
Processed 560 frames | FPS=41.77
Processed 58

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import numpy as np
from google.colab import files
import time

# =====================================================
# UPLOAD VIDEO
# =====================================================

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

print("Video:", video_path)

OUTPUT_VIDEO = "tablet_defect_output.mp4"

MIN_TABLET_AREA = 3000

CHIP_BRIGHTNESS = 180
CAP_TEXTURE = 35

chip_kernel = np.ones((5,5),np.uint8)
cap_kernel = np.ones((9,9),np.uint8)


# =====================================================
# PREPROCESS
# =====================================================

def preprocess(frame):

    blur = cv2.GaussianBlur(
        frame,
        (5,5),
        0
    )

    hsv = cv2.cvtColor(
        blur,
        cv2.COLOR_BGR2HSV
    )

    gray = hsv[:,:,2]

    return gray


# =====================================================
# TABLET SEGMENTATION
# =====================================================

def segment_tablet(gray):

    _,mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY+cv2.THRESH_OTSU
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        np.ones((9,9),np.uint8)
    )

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours)==0:
        return None,None

    contour = max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(contour)<MIN_TABLET_AREA:
        return None,None

    clean = np.zeros_like(mask)

    cv2.drawContours(
        clean,
        [contour],
        -1,
        255,
        -1
    )

    return clean, contour


# =====================================================
# CHIP DETECTION
# =====================================================

def detect_chip(frame, tablet_mask):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # only examine boundary

    inner = cv2.erode(
        tablet_mask,
        np.ones((15,15),np.uint8),
        iterations=1
    )

    boundary = cv2.subtract(
        tablet_mask,
        inner
    )

    bright = cv2.threshold(
        gray,
        CHIP_BRIGHTNESS,
        255,
        cv2.THRESH_BINARY
    )[1]

    chip = cv2.bitwise_and(
        bright,
        boundary
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_OPEN,
        chip_kernel
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_CLOSE,
        chip_kernel
    )

    return chip


# =====================================================
# CAP DETECTION
# =====================================================

def detect_cap(frame, tablet_mask):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    texture = cv2.Laplacian(
        gray,
        cv2.CV_64F
    )

    texture = np.abs(
        texture
    )

    texture = np.uint8(
        texture
    )

    texture = cv2.bitwise_and(
        texture,
        tablet_mask
    )

    cap = cv2.threshold(
        texture,
        CAP_TEXTURE,
        255,
        cv2.THRESH_BINARY
    )[1]

    cap = cv2.morphologyEx(
        cap,
        cv2.MORPH_CLOSE,
        cap_kernel
    )

    cap = cv2.morphologyEx(
        cap,
        cv2.MORPH_OPEN,
        cap_kernel
    )

    return cap


# =====================================================
# DRAW DETECTIONS
# =====================================================

def draw_regions(frame, mask, label, color):

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if area < 80:
            continue

        x,y,w,h = cv2.boundingRect(c)

        cv2.rectangle(
            frame,
            (x,y),
            (x+w,y+h),
            color,
            3
        )

        cv2.putText(
            frame,
            label,
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

        overlay = frame.copy()

        cv2.drawContours(
            overlay,
            [c],
            -1,
            color,
            -1
        )

        frame = cv2.addWeighted(
            overlay,
            0.35,
            frame,
            0.65,
            0
        )

    return frame


# =====================================================
# PROCESS
# =====================================================

def process(frame):

    gray = preprocess(frame)

    tablet_mask, contour = segment_tablet(
        gray
    )

    if contour is None:
        return frame

    cv2.drawContours(
        frame,
        [contour],
        -1,
        (0,255,0),
        3
    )

    chip_mask = detect_chip(
        frame,
        tablet_mask
    )

    cap_mask = detect_cap(
        frame,
        tablet_mask
    )

    frame = draw_regions(
        frame,
        chip_mask,
        "CHIP",
        (0,0,255)
    )

    frame = draw_regions(
        frame,
        cap_mask,
        "CAP",
        (255,0,0)
    )

    return frame


# =====================================================
# VIDEO
# =====================================================

cap = cv2.VideoCapture(
    video_path
)

fps = int(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width,height)
)

count = 0
start = time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    out = process(frame)

    writer.write(out)

    count += 1

    if count % 20 == 0:

        elapsed = time.time() - start

        print(
            f"Processed {count} frames | FPS={count/elapsed:.2f}"
        )

cap.release()
writer.release()

print("\nDONE")

files.download(
    OUTPUT_VIDEO
)

Saving input1.mp4 to input1.mp4
Video: input1.mp4
Processed 20 frames | FPS=12.16
Processed 40 frames | FPS=9.95
Processed 60 frames | FPS=10.25
Processed 80 frames | FPS=10.91
Processed 100 frames | FPS=11.36
Processed 120 frames | FPS=11.52
Processed 140 frames | FPS=11.39
Processed 160 frames | FPS=11.54
Processed 180 frames | FPS=10.86
Processed 200 frames | FPS=10.84
Processed 220 frames | FPS=10.99
Processed 240 frames | FPS=10.95
Processed 260 frames | FPS=11.09
Processed 280 frames | FPS=11.23
Processed 300 frames | FPS=11.32
Processed 320 frames | FPS=11.30
Processed 340 frames | FPS=11.16
Processed 360 frames | FPS=11.25
Processed 380 frames | FPS=11.34
Processed 400 frames | FPS=11.43
Processed 420 frames | FPS=11.50
Processed 440 frames | FPS=11.56
Processed 460 frames | FPS=11.64
Processed 480 frames | FPS=11.63
Processed 500 frames | FPS=11.51
Processed 520 frames | FPS=11.59
Processed 540 frames | FPS=11.66
Processed 560 frames | FPS=11.73
Processed 580 frames | FPS=11.7

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import numpy as np
import time
from google.colab import files


# =====================================================
# PATHS
# =====================================================

VIDEO_PATH = "input1_crop.mp4"

# Example:
# VIDEO_PATH="/content/drive/MyDrive/input1_crop.mp4"

OUTPUT_VIDEO = "tablet_defect_output.mp4"


# =====================================================
# PARAMETERS
# =====================================================

MIN_TABLET_AREA = 2500

CHIP_BRIGHTNESS = 175

CAP_TEXTURE = 28

BOUNDARY_WIDTH = 18

MIN_DEFECT_AREA = 80


# =====================================================
# PREPROCESS
# =====================================================

def preprocess(frame):

    blur = cv2.GaussianBlur(
        frame,
        (5,5),
        0
    )

    hsv = cv2.cvtColor(
        blur,
        cv2.COLOR_BGR2HSV
    )

    gray = hsv[:,:,2]

    return gray


# =====================================================
# TABLET SEGMENTATION
# =====================================================

def segment_tablet(gray):

    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY +
        cv2.THRESH_OTSU
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        np.ones((11,11),np.uint8)
    )

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours)==0:
        return None,None

    contour = max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(contour)<MIN_TABLET_AREA:
        return None,None

    clean = np.zeros_like(mask)

    cv2.drawContours(
        clean,
        [contour],
        -1,
        255,
        -1
    )

    return clean, contour


# =====================================================
# CHIP DETECTION
# =====================================================

def detect_chip(frame, tablet_mask):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    inner = cv2.erode(
        tablet_mask,
        np.ones(
            (
                BOUNDARY_WIDTH,
                BOUNDARY_WIDTH
            ),
            np.uint8
        ),
        iterations=1
    )

    boundary = cv2.subtract(
        tablet_mask,
        inner
    )

    bright = cv2.threshold(
        gray,
        CHIP_BRIGHTNESS,
        255,
        cv2.THRESH_BINARY
    )[1]

    chip = cv2.bitwise_and(
        boundary,
        bright
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_OPEN,
        np.ones((5,5),np.uint8)
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_CLOSE,
        np.ones((7,7),np.uint8)
    )

    return chip


# =====================================================
# CAP DETECTION
# =====================================================

def detect_cap(frame, tablet_mask):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    lap = cv2.Laplacian(
        gray,
        cv2.CV_64F
    )

    lap = np.abs(
        lap
    )

    lap = np.uint8(
        lap
    )

    lap = cv2.bitwise_and(
        lap,
        tablet_mask
    )

    cap = cv2.threshold(
        lap,
        CAP_TEXTURE,
        255,
        cv2.THRESH_BINARY
    )[1]

    cap = cv2.morphologyEx(
        cap,
        cv2.MORPH_CLOSE,
        np.ones((11,11),np.uint8)
    )

    cap = cv2.morphologyEx(
        cap,
        cv2.MORPH_OPEN,
        np.ones((5,5),np.uint8)
    )

    return cap


# =====================================================
# DRAW DEFECT
# =====================================================

def draw_defect(
    frame,
    defect_mask,
    label,
    color
):

    contours,_ = cv2.findContours(
        defect_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        area = cv2.contourArea(c)

        if area < MIN_DEFECT_AREA:
            continue

        x,y,w,h = cv2.boundingRect(c)

        overlay = frame.copy()

        cv2.drawContours(
            overlay,
            [c],
            -1,
            color,
            -1
        )

        frame = cv2.addWeighted(
            overlay,
            0.35,
            frame,
            0.65,
            0
        )

        cv2.rectangle(
            frame,
            (x,y),
            (x+w,y+h),
            color,
            3
        )

        cv2.putText(
            frame,
            label,
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

    return frame


# =====================================================
# PROCESS FRAME
# =====================================================

def process(frame):

    gray = preprocess(
        frame
    )

    mask, contour = segment_tablet(
        gray
    )

    if contour is None:
        return frame

    cv2.drawContours(
        frame,
        [contour],
        -1,
        (0,255,0),
        3
    )

    chip = detect_chip(
        frame,
        mask
    )

    cap = detect_cap(
        frame,
        mask
    )

    frame = draw_defect(
        frame,
        chip,
        "CHIP",
        (0,0,255)
    )

    frame = draw_defect(
        frame,
        cap,
        "CAP",
        (255,0,0)
    )

    return frame


# =====================================================
# VIDEO
# =====================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

fps = int(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width,height)
)

count = 0

start = time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    result = process(
        frame
    )

    writer.write(
        result
    )

    count += 1

    if count % 30 == 0:

        elapsed = (
            time.time()
            -
            start
        )

        print(
            f"Frames={count} | FPS={count/elapsed:.1f}"
        )

cap.release()

writer.release()


print("\nProcessing Completed")

files.download(
    OUTPUT_VIDEO
)

Frames=30 | FPS=38.7
Frames=60 | FPS=35.8
Frames=90 | FPS=35.6
Frames=120 | FPS=36.0
Frames=150 | FPS=34.4
Frames=180 | FPS=33.5
Frames=210 | FPS=33.3
Frames=240 | FPS=34.2
Frames=270 | FPS=34.6
Frames=300 | FPS=35.2
Frames=330 | FPS=35.6
Frames=360 | FPS=35.9
Frames=390 | FPS=36.3
Frames=420 | FPS=36.6
Frames=450 | FPS=36.9
Frames=480 | FPS=37.1
Frames=510 | FPS=37.2
Frames=540 | FPS=37.3
Frames=570 | FPS=37.4
Frames=600 | FPS=37.5
Frames=630 | FPS=36.9
Frames=660 | FPS=36.5
Frames=690 | FPS=36.0
Frames=720 | FPS=36.1
Frames=750 | FPS=36.2
Frames=780 | FPS=36.3

Processing Completed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Variation 2**

In [ ]:
import cv2
import numpy as np
import time
from google.colab import files


# =====================================================
# PATH
# =====================================================

VIDEO_PATH = "input1_crop.mp4"

OUTPUT_VIDEO = "tablet_segmentation_output.mp4"


# =====================================================
# PARAMETERS
# =====================================================

MIN_TABLET_AREA = 2500

CHIP_BRIGHTNESS = 175

CAP_TEXTURE = 28

BOUNDARY_WIDTH = 18

MIN_DEFECT_AREA = 120

ALPHA = 0.45


# =====================================================
# PREPROCESS
# =====================================================

def preprocess(frame):

    blur = cv2.GaussianBlur(
        frame,
        (5,5),
        0
    )

    hsv = cv2.cvtColor(
        blur,
        cv2.COLOR_BGR2HSV
    )

    return hsv[:,:,2]


# =====================================================
# TABLET SEGMENTATION
# =====================================================

def segment_tablet(gray):

    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY +
        cv2.THRESH_OTSU
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        np.ones((11,11),np.uint8)
    )

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours)==0:
        return None,None

    contour=max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(
        contour
    ) < MIN_TABLET_AREA:

        return None,None

    clean=np.zeros_like(mask)

    cv2.drawContours(
        clean,
        [contour],
        -1,
        255,
        -1
    )

    return clean,contour


# =====================================================
# CHIP MASK
# =====================================================

def detect_chip(frame, tablet):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    inner = cv2.erode(
        tablet,
        np.ones(
            (
                BOUNDARY_WIDTH,
                BOUNDARY_WIDTH
            ),
            np.uint8
        ),
        iterations=1
    )

    ring = cv2.subtract(
        tablet,
        inner
    )

    white = cv2.threshold(
        gray,
        CHIP_BRIGHTNESS,
        255,
        cv2.THRESH_BINARY
    )[1]

    chip = cv2.bitwise_and(
        ring,
        white
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_CLOSE,
        np.ones((7,7),np.uint8)
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_OPEN,
        np.ones((5,5),np.uint8)
    )

    return chip


# =====================================================
# CAP MASK
# =====================================================

def detect_cap(frame, tablet):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    lap = cv2.Laplacian(
        gray,
        cv2.CV_64F
    )

    lap = np.uint8(
        np.abs(
            lap
        )
    )

    lap = cv2.bitwise_and(
        lap,
        tablet
    )

    cap = cv2.threshold(
        lap,
        CAP_TEXTURE,
        255,
        cv2.THRESH_BINARY
    )[1]

    cap = cv2.morphologyEx(
        cap,
        cv2.MORPH_CLOSE,
        np.ones((11,11),np.uint8)
    )

    cap = cv2.morphologyEx(
        cap,
        cv2.MORPH_OPEN,
        np.ones((5,5),np.uint8)
    )

    return cap


# =====================================================
# DRAW SEGMENTATION
# =====================================================

def draw_mask(
    image,
    mask,
    color
):

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    overlay = image.copy()

    for c in contours:

        if cv2.contourArea(
            c
        ) < MIN_DEFECT_AREA:

            continue

        cv2.drawContours(
            overlay,
            [c],
            -1,
            color,
            -1
        )

        cv2.drawContours(
            image,
            [c],
            -1,
            color,
            2
        )

    return cv2.addWeighted(
        overlay,
        ALPHA,
        image,
        1-ALPHA,
        0
    )


# =====================================================
# PROCESS
# =====================================================

def process(frame):

    gray = preprocess(
        frame
    )

    tablet, contour = segment_tablet(
        gray
    )

    if contour is None:

        return frame

    cv2.drawContours(
        frame,
        [contour],
        -1,
        (0,255,0),
        3
    )

    chip = detect_chip(
        frame,
        tablet
    )

    cap = detect_cap(
        frame,
        tablet
    )

    frame = draw_mask(
        frame,
        chip,
        (0,0,255)
    )

    frame = draw_mask(
        frame,
        cap,
        (255,0,0)
    )

    return frame


# =====================================================
# VIDEO LOOP
# =====================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

fps = int(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)

w = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

h = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

writer = cv2.VideoWriter(

    OUTPUT_VIDEO,

    cv2.VideoWriter_fourcc(
        *'mp4v'
    ),

    fps,

    (w,h)

)

count = 0

start = time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    out = process(
        frame
    )

    writer.write(
        out
    )

    count += 1

    if count % 30 == 0:

        fps_now = count / (
            time.time()
            -
            start
        )

        print(
            f"Processed {count} | FPS={fps_now:.1f}"
        )

cap.release()

writer.release()

print("\nSaved:", OUTPUT_VIDEO)

files.download(
    OUTPUT_VIDEO
)

Processed 30 | FPS=38.8
Processed 60 | FPS=37.5
Processed 90 | FPS=37.4
Processed 120 | FPS=37.6
Processed 150 | FPS=35.3
Processed 180 | FPS=33.9
Processed 210 | FPS=33.1
Processed 240 | FPS=33.9
Processed 270 | FPS=34.4
Processed 300 | FPS=34.8
Processed 330 | FPS=35.2
Processed 360 | FPS=35.5
Processed 390 | FPS=35.8
Processed 420 | FPS=36.0
Processed 450 | FPS=36.3
Processed 480 | FPS=36.4
Processed 510 | FPS=36.6
Processed 540 | FPS=36.7
Processed 570 | FPS=36.9
Processed 600 | FPS=37.0
Processed 630 | FPS=36.5
Processed 660 | FPS=36.1
Processed 690 | FPS=35.6
Processed 720 | FPS=35.8
Processed 750 | FPS=35.9
Processed 780 | FPS=36.0

Saved: tablet_segmentation_output.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Variation 3**

In [ ]:
import cv2
import numpy as np
import time
from google.colab import files


# ====================================================
# VIDEO
# ====================================================

VIDEO_PATH = "input1_crop.mp4"

OUTPUT_VIDEO = "tablet_defect_segmentation.mp4"


# ====================================================
# PARAMETERS
# ====================================================

MIN_TABLET_AREA = 2500

BOUNDARY_WIDTH = 22

MIN_DEFECT_AREA = 250

MASK_ALPHA = 0.45


# ====================================================
# PREPROCESS
# ====================================================

def preprocess(frame):

    blur = cv2.GaussianBlur(
        frame,
        (5,5),
        0
    )

    hsv = cv2.cvtColor(
        blur,
        cv2.COLOR_BGR2HSV
    )

    return hsv[:,:,2]


# ====================================================
# TABLET SEGMENTATION
# ====================================================

def segment_tablet(gray):

    _,mask = cv2.threshold(

        gray,

        0,

        255,

        cv2.THRESH_BINARY +
        cv2.THRESH_OTSU

    )

    mask = cv2.morphologyEx(

        mask,

        cv2.MORPH_CLOSE,

        np.ones((11,11),np.uint8)

    )

    contours,_ = cv2.findContours(

        mask,

        cv2.RETR_EXTERNAL,

        cv2.CHAIN_APPROX_SIMPLE

    )

    if len(contours)==0:
        return None,None

    contour=max(
        contours,
        key=cv2.contourArea
    )

    if cv2.contourArea(
        contour
    ) < MIN_TABLET_AREA:

        return None,None

    tablet=np.zeros_like(mask)

    cv2.drawContours(

        tablet,

        [contour],

        -1,

        255,

        -1

    )

    return tablet, contour


# ====================================================
# CHIP DETECTION
# ====================================================

def detect_chip(frame, tablet):

    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    h,s,v = cv2.split(
        hsv
    )

    inner = cv2.erode(

        tablet,

        np.ones(
            (
                BOUNDARY_WIDTH,
                BOUNDARY_WIDTH
            ),
            np.uint8
        ),

        iterations=1

    )

    boundary = cv2.subtract(
        tablet,
        inner
    )

    white = np.logical_and(
        s < 60,
        v > 170
    )

    white = (
        white.astype(
            np.uint8
        )
        *
        255
    )

    chip = cv2.bitwise_and(
        white,
        boundary
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_CLOSE,
        np.ones((9,9),np.uint8)
    )

    chip = cv2.morphologyEx(
        chip,
        cv2.MORPH_OPEN,
        np.ones((5,5),np.uint8)
    )

    return chip


# ====================================================
# CAP DETECTION
# ====================================================
def detect_cap(frame, tablet):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # improve local contrast
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    gray = clahe.apply(
        gray
    )

    # blackhat → dark crack extraction
    blackhat = cv2.morphologyEx(

        gray,

        cv2.MORPH_BLACKHAT,

        np.ones(
            (15,15),
            np.uint8
        )

    )

    blackhat = cv2.bitwise_and(
        blackhat,
        tablet
    )

    # strong responses only

    _, cap = cv2.threshold(

        blackhat,

        18,

        255,

        cv2.THRESH_BINARY

    )

    cap = cv2.morphologyEx(

        cap,

        cv2.MORPH_OPEN,

        np.ones(
            (3,3),
            np.uint8
        )

    )

    cap = cv2.morphologyEx(

        cap,

        cv2.MORPH_CLOSE,

        np.ones(
            (9,9),
            np.uint8
        )

    )

    # remove large smooth regions

    contours,_ = cv2.findContours(
        cap,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    filtered = np.zeros_like(
        cap
    )

    for c in contours:

        area = cv2.contourArea(
            c
        )

        if area < 100:
            continue

        x,y,w,h = cv2.boundingRect(
            c
        )

        ratio = max(
            w,
            h
        ) / (
            min(
                w,
                h
            ) + 1
        )

        # keep crack-like regions

        if ratio > 1.5:

            cv2.drawContours(

                filtered,

                [c],

                -1,

                255,

                -1

            )

    return filtered



# ====================================================
# DRAW SEGMENTATION
# ====================================================

def draw_mask(
    image,
    mask,
    color
):

    contours,_ = cv2.findContours(

        mask,

        cv2.RETR_EXTERNAL,

        cv2.CHAIN_APPROX_SIMPLE

    )

    overlay = image.copy()

    for c in contours:

        if cv2.contourArea(
            c
        ) < MIN_DEFECT_AREA:

            continue

        cv2.drawContours(

            overlay,

            [c],

            -1,

            color,

            -1

        )

        cv2.drawContours(

            image,

            [c],

            -1,

            color,

            2

        )

    return cv2.addWeighted(

        overlay,

        MASK_ALPHA,

        image,

        1-MASK_ALPHA,

        0

    )


# ====================================================
# FRAME PROCESS
# ====================================================

def process(frame):

    gray = preprocess(
        frame
    )

    tablet, contour = segment_tablet(
        gray
    )

    if contour is None:

        return frame

    cv2.drawContours(

        frame,

        [contour],

        -1,

        (0,255,0),

        3

    )

    chip = detect_chip(
        frame,
        tablet
    )

    cap = detect_cap(
        frame,
        tablet
    )

    frame = draw_mask(

        frame,

        chip,

        (0,0,255)

    )

    frame = draw_mask(

        frame,

        cap,

        (255,0,0)

    )

    return frame


# ====================================================
# VIDEO LOOP
# ====================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

fps = int(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)

w = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

h = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

writer = cv2.VideoWriter(

    OUTPUT_VIDEO,

    cv2.VideoWriter_fourcc(
        *'mp4v'
    ),

    fps,

    (w,h)

)

count=0

start=time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    out = process(
        frame
    )

    writer.write(
        out
    )

    count += 1

    if count % 30 == 0:

        elapsed = (
            time.time()
            -
            start
        )

        print(
            f"{count} frames | FPS={count/elapsed:.1f}"
        )

cap.release()

writer.release()

print("\nFinished")

files.download(
    OUTPUT_VIDEO
)

30 frames | FPS=21.6
60 frames | FPS=20.7
90 frames | FPS=22.5
120 frames | FPS=24.3
150 frames | FPS=25.6
180 frames | FPS=26.5
210 frames | FPS=27.2
240 frames | FPS=27.8
270 frames | FPS=28.3
300 frames | FPS=28.7
330 frames | FPS=29.0
360 frames | FPS=29.2
390 frames | FPS=29.4
420 frames | FPS=28.8
450 frames | FPS=28.3
480 frames | FPS=28.3
510 frames | FPS=28.5
540 frames | FPS=28.7
570 frames | FPS=28.9
600 frames | FPS=29.1
630 frames | FPS=29.2
660 frames | FPS=29.4
690 frames | FPS=29.5
720 frames | FPS=29.6
750 frames | FPS=29.7
780 frames | FPS=29.8

Finished


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>